<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/DataAnalyzer001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.fft import fft
from scipy.signal import find_peaks
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, r2_score

# ==============================
# Configuration
# ==============================

class Config:
    # Sweep defaults (replace with real)
    z_min = 1e-3
    z_max = 1e-2
    n_points = 120
    ring_size = 64

    # Analysis parameters
    peak_prominence = 0.08
    histogram_bins = 60
    figsize = (18, 10)

    # Convergence study grids
    convergence_n_points = [60, 120, 240]
    convergence_ring_sizes = [32, 64, 128]

    # Bootstrap for generation clustering test
    n_bootstraps = 1000

    # Mass calibration (optional)
    calib_target_mass = None
    calib_anchor_index = None


# ==============================
# Feature construction
# ==============================

def add_force_strength(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    def ring_rms(r):
        r = np.asarray(r, dtype=float)
        return float(np.sqrt(np.mean(r**2)))
    out['force_strength'] = out['curvature_ring'].apply(ring_rms)
    return out

def add_mass_hypotheses(df: pd.DataFrame, mass_unit: float = 1.0) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-12

    out['m_inv_z'] = mass_unit / (out['z_depth'] + eps)

    def ring_energy(r):
        r = np.asarray(r, dtype=float)
        return float(np.sum(r**2))
    out['m_ring_energy'] = out['curvature_ring'].apply(ring_energy)

    out['m_balance'] = out['memory_amp'] / (out['max_dip'] + eps)

    for col in ['m_inv_z', 'm_ring_energy', 'm_balance']:
        x = out[col].values
        out[col + '_zscore'] = (x - np.mean(x)) / (np.std(x) + eps)
    return out

def add_stability_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-12
    s, m = out['max_dip'], out['memory_amp']
    s_norm = (s - s.min()) / (s.max() - s.min() + eps)
    m_norm = (m - m.min()) / (m.max() - m.min() + eps)

    out['stability_exp'] = m_norm * np.exp(-s_norm)
    out['stability_ratio'] = m / (s + eps)
    out['stability_logistic'] = m_norm / (1.0 + np.exp(5.0 * (s_norm - 0.5)))
    return out

def add_ring_metrics(df: pd.DataFrame, peak_prominence: float = 0.08) -> pd.DataFrame:
    out = df.copy()
    gen_counts, peak_lists = [], []
    for _, row in out.iterrows():
        ring = np.asarray(row['curvature_ring'], dtype=float)
        spec = np.abs(fft(ring - ring.mean()))[:ring.size // 2]
        spec_norm = spec / (spec.max() + 1e-12)
        peaks, _ = find_peaks(spec_norm, prominence=peak_prominence)
        gen_counts.append(len(peaks))
        peak_lists.append(peaks.tolist())
    out['generation_count'] = gen_counts
    out['ring_peak_indices'] = peak_lists
    return out

def add_dipole_mode(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dip = []
    for _, row in out.iterrows():
        er = row.get('entropy_ring', None)
        if er is None:
            dip.append(np.nan)
            continue
        arr = np.asarray(er, dtype=float)
        spec = np.abs(fft(arr - arr.mean()))
        dip.append(spec[1] / (spec.max() + 1e-12))
    out['dipole_mode_strength'] = dip
    return out


# ==============================
# Visualization (atlas)
# ==============================

def atlas_hypothesis(df: pd.DataFrame, bins: int = 60, figsize=(18,10)):
    fig, axs = plt.subplots(2, 3, figsize=figsize)

    # Mass hypotheses
    axs[0,0].hist(df['m_inv_z'], bins=bins, alpha=0.7, label='m_inv_z')
    axs[0,0].hist(df['m_ring_energy'], bins=bins, alpha=0.7, label='m_ring_energy')
    axs[0,0].hist(df['m_balance'], bins=bins, alpha=0.7, label='m_balance')
    axs[0,0].set_title('Mass hypotheses')
    axs[0,0].set_xlabel('Value'); axs[0,0].set_ylabel('Count'); axs[0,0].legend()

    # Dip vs lifetime (color = stability_exp)
    sc = axs[0,1].scatter(df['max_dip'], df['lifetime'], c=df['stability_exp'], cmap='viridis', s=35)
    axs[0,1].set_title('Dip depth vs Lifetime (color = stability_exp)')
    axs[0,1].set_xlabel('|S| dip depth'); axs[0,1].set_ylabel('Lifetime')
    c1 = fig.colorbar(sc, ax=axs[0,1]); c1.set_label('stability_exp')

    # Force vs ring-energy mass (color = m_inv_z_zscore)
    sc2 = axs[0,2].scatter(df['force_strength'], df['m_ring_energy'], c=df['m_inv_z_zscore'], cmap='plasma', s=35)
    axs[0,2].set_title('Force strength vs ring-energy mass')
    axs[0,2].set_xlabel('Curvature RMS'); axs[0,2].set_ylabel('m_ring_energy')
    c2 = fig.colorbar(sc2, ax=axs[0,2]); c2.set_label('m_inv_z_zscore')

    # Stability metrics vs lifetime
    axs[1,0].scatter(df['stability_exp'], df['lifetime'], s=25, alpha=0.8, label='exp')
    axs[1,0].scatter(df['stability_ratio'], df['lifetime'], s=25, alpha=0.6, label='ratio')
    axs[1,0].scatter(df['stability_logistic'], df['lifetime'], s=25, alpha=0.6, label='logistic')
    axs[1,0].set_title('Do stability metrics predict lifetime?')
    axs[1,0].set_xlabel('Stability metric'); axs[1,0].set_ylabel('Lifetime'); axs[1,0].legend()

    # Generation counts (uncapped)
    counts = df['generation_count'].value_counts().sort_index()
    axs[1,1].bar(counts.index.astype(int), counts.values, color='indianred', alpha=0.85)
    axs[1,1].set_title('Generation counts (FFT peaks)')
    axs[1,1].set_xlabel('Peak count'); axs[1,1].set_ylabel('States')

    # Dipole mode vs m_inv_z (color = force_strength)
    sc3 = axs[1,2].scatter(df['m_inv_z'], df['dipole_mode_strength'], c=df['force_strength'], cmap='magma', s=30)
    axs[1,2].set_title('Dipole mode strength vs m_inv_z')
    axs[1,2].set_xlabel('m_inv_z'); axs[1,2].set_ylabel('Dipole mode strength')
    c3 = fig.colorbar(sc3, ax=axs[1,2]); c3.set_label('Curvature RMS')

    plt.tight_layout()
    plt.show()


# ==============================
# Analysis helpers
# ==============================

def correlation_report(df: pd.DataFrame, target='lifetime'):
    cols = ['m_inv_z','m_ring_energy','m_balance','stability_exp','stability_ratio','stability_logistic']
    print(f'Correlation to {target}:')
    for c in cols:
        x, y = df[c].values, df[target].values
        r = np.corrcoef(x, y)[0,1]
        print(f'- {c}: r = {r:.3f}')

def test_generation_clustering(df: pd.DataFrame, n_bootstraps: int = 1000):
    observed = df['generation_count'].values
    min_c, max_c = int(observed.min()), int(observed.max())
    obs_var = float(np.var(observed))
    null_vars = []
    rng = np.random.default_rng()
    for _ in range(n_bootstraps):
        null_sample = rng.integers(low=min_c, high=max_c+1, size=len(observed))
        null_vars.append(float(np.var(null_sample)))
    p_value = np.mean([nv >= obs_var for nv in null_vars])
    print(f'Generation clustering variance test: p = {p_value:.4f}')
    if p_value < 0.05:
        print('→ Significant clustering (reject uniform).')
    else:
        print('→ Consistent with uniform distribution.')

def eliminate_hypotheses(df: pd.DataFrame):
    tests = {}
    # Clustering quality for mass hypotheses
    for h in ['m_inv_z','m_ring_energy','m_balance']:
        X = df[[h]].values
        # Try k=2..5 and keep best silhouette
        best_s = -1.0
        for k in [2,3,4,5]:
            km = KMeans(n_clusters=k, n_init=10, random_state=42)
            labels = km.fit_predict(X)
            try:
                s = silhouette_score(X, labels)
                best_s = max(best_s, s)
            except Exception:
                pass
        tests[h] = {'best_silhouette': best_s}

    # Predictive power for stability metrics
    for s in ['stability_exp','stability_ratio','stability_logistic']:
        y = df['lifetime'].values
        X = df[[s]].values
        # Simple linear fit proxy via r2_score of mean model vs metric mapping
        # Use a linear regression in spirit: y_hat = a * metric + b (a,b via least squares)
        x = X[:,0]
        A = np.vstack([x, np.ones_like(x)]).T
        a, b = np.linalg.lstsq(A, y, rcond=None)[0]
        y_hat = a*x + b
        r2 = r2_score(y, y_hat)
        tests[s] = {'r2': r2}

    print('Hypothesis test results:')
    for name, metrics in tests.items():
        print(f'- {name}: {metrics}')

    print('\nEliminations:')
    for h in ['m_inv_z','m_ring_energy','m_balance']:
        if tests[h]['best_silhouette'] < 0.3:
            print(f'  • {h}: poor clustering (silhouette < 0.3)')
    for s in ['stability_exp','stability_ratio','stability_logistic']:
        if tests[s]['r2'] < 0.1:
            print(f'  • {s}: low predictive power (R² < 0.1)')

def convergence_study(run_sweep_fn):
    """
    run_sweep_fn(n_points, ring_size) -> success_df
    """
    records = []
    for n_points in Config.convergence_n_points:
        for ring_size in Config.convergence_ring_sizes:
            df = run_sweep_fn(n_points=n_points, ring_size=ring_size)
            decoded = decode_pipeline(df, peak_prominence=Config.peak_prominence, mass_unit=1.0, show_plots=False)
            r_stab = np.corrcoef(decoded['stability_exp'], decoded['lifetime'])[0,1]
            records.append({
                'n_points': n_points,
                'ring_size': ring_size,
                'mean_gen_count': float(decoded['generation_count'].mean()),
                'mass_clustering_silhouette_m_inv_z': silhouette_score(decoded[['m_inv_z']].values,
                                                                      KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(decoded[['m_inv_z']].values)),
                'stability_exp_corr': float(r_stab)
            })
    res = pd.DataFrame(records)
    print('Convergence study summary:')
    print(res)
    return res


# ==============================
# Orchestration
# ==============================

def decode_pipeline(success_df: pd.DataFrame, peak_prominence: float = 0.08, mass_unit: float = 1.0, show_plots: bool = True):
    out = success_df.copy()
    out = add_force_strength(out)
    out = add_mass_hypotheses(out, mass_unit=mass_unit)
    out = add_stability_metrics(out)
    out = add_ring_metrics(out, peak_prominence=peak_prominence)
    out = add_dipole_mode(out)
    if show_plots:
        atlas_hypothesis(out, bins=Config.histogram_bins, figsize=Config.figsize)
    return out

def calibrate_mass_unit(df: pd.DataFrame, target_mass: float, anchor_index: int) -> float:
    z_anchor = float(df.iloc[anchor_index]['z_depth'])
    return target_mass * z_anchor


# ==============================
# Synthetic sweep (for testing only)
# Replace with your real sweep
# ==============================

def generate_fake_sweep(n_points=Config.n_points, ring_size=Config.ring_size):
    rng = np.random.default_rng()
    z_values = np.linspace(Config.z_min, Config.z_max, n_points)
    rows = []
    for z in z_values:
        S_dip = np.exp(-z * 900) + 0.08 * rng.random()
        tau = 120 * np.exp(-z * 150) + 15 * rng.random()
        theta = np.linspace(0, 2*np.pi, ring_size, endpoint=False)
        ring = (0.6*np.sin(1*theta + rng.random()) +
                0.4*np.sin(2*theta + rng.random()) +
                0.25*np.sin(3*theta + rng.random()) +
                0.15*rng.normal(0, 1, size=ring_size))
        entropy_ring = (0.7*np.sin(1*theta + rng.random()) +
                        0.2*np.sin(2*theta + rng.random()) +
                        0.1*rng.normal(0, 1, size=ring_size))
        M_amp = 10 * rng.random()
        rows.append({
            'z_depth': z,
            'max_dip': float(S_dip),
            'lifetime': float(tau),
            'curvature_ring': np.asarray(ring, float),
            'entropy_ring': np.asarray(entropy_ring, float),
            'memory_amp': float(M_amp),
        })
    return pd.DataFrame(rows)

def run_sweep_for_convergence(n_points: int, ring_size: int) -> pd.DataFrame:
    # Same synthetic generator but parameterized; replace with real engine
    rng = np.random.default_rng()
    z_values = np.linspace(Config.z_min, Config.z_max, n_points)
    rows = []
    theta = np.linspace(0, 2*np.pi, ring_size, endpoint=False)
    for z in z_values:
        S_dip = np.exp(-z * 900) + 0.08 * rng.random()
        tau = 120 * np.exp(-z * 150) + 15 * rng.random()
        ring = (0.6*np.sin(1*theta + rng.random()) +
                0.4*np.sin(2*theta + rng.random()) +
                0.25*np.sin(3*theta + rng.random()) +
                0.15*rng.normal(0, 1, size=ring_size))
        entropy_ring = (0.7*np.sin(1*theta + rng.random()) +
                        0.2*np.sin(2*theta + rng.random()) +
                        0.1*rng.normal(0, 1, size=ring_size))
        M_amp = 10 * rng.random()
        rows.append({
            'z_depth': z,
            'max_dip': float(S_dip),
            'lifetime': float(tau),
            'curvature_ring': np.asarray(ring, float),
            'entropy_ring': np.asarray(entropy_ring, float),
            'memory_amp': float(M_amp),
        })
    return pd.DataFrame(rows)


# ==============================
# Execution example
# ==============================

if __name__ == "__main__":
    # Use real sweep DataFrame here:
    success_df = generate_fake_sweep()

    # Optional mass calibration
    if Config.calib_target_mass is not None and Config.calib_anchor_index is not None:
        mass_unit = calibrate_mass_unit(success_df, Config.calib_target_mass, Config.calib_anchor_index)
    else:
        mass_unit = 1.0

    decoded = decode_pipeline(success_df, peak_prominence=Config.peak_prominence, mass_unit=mass_unit)

    # Reports
    correlation_report(decoded)
    test_generation_clustering(decoded, n_bootstraps=Config.n_bootstraps)
    eliminate_hypotheses(decoded)

    # Convergence study (synthetic example; wire to real run_sweep_fn when ready)
    convergence_study(run_sweep_for_convergence)

    # Quick view
    cols = ['z_depth','max_dip','lifetime','force_strength',
            'm_inv_z','m_ring_energy','m_balance',
            'stability_exp','stability_ratio','stability_logistic',
            'generation_count','dipole_mode_strength']
    print(decoded[cols].head())
